# Point Forecast Reference (Linear Regression)
This notebook is a simple reference for point forecasting only.

Models:
- `naive_24`: Linear Regression with `lag_24`
- `naive_168`: Linear Regression with `lag_168`
- `naive_24_168_combined`: Linear Regression with both lags

It uses `params.yaml` and `dvc.yaml` for paths and split settings.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))



In [2]:
# Load params/dvc config
params_path = PROJECT_ROOT / "params.yaml"
dvc_path = PROJECT_ROOT / "dvc.yaml"

params = yaml.safe_load(params_path.read_text(encoding="utf-8"))
dvc_cfg = yaml.safe_load(dvc_path.read_text(encoding="utf-8"))
cfg = params["model_training"]

# Keep coherence with pipeline config
if dvc_cfg["stages"]["validate_data"]["outs"][0] != "${validation.output_path}":
    raise ValueError("Unexpected validate_data output in dvc.yaml")
if cfg["input_path"] != params["validation"]["output_path"]:
    raise ValueError("model_training.input_path must equal validation.output_path")

input_path = PROJECT_ROOT / cfg["input_path"]
target_col = cfg["target_col"]
time_col = cfg["time_col"]
train_fraction = float(cfg["split"]["train_fraction"])

print(f"Input: {input_path}")
print(f"Target: {target_col} | Time: {time_col}")
print(f"Train fraction: {train_fraction}")


Input: /home/nilima/Desktop/portfolio/probabilistic-energy-forecasting-ML/data/validated/validated_data.parquet
Target: residual_load_mwh | Time: timestamp
Train fraction: 0.8


In [3]:
# Load data and build lag features

df = pd.read_parquet(input_path).copy()
df[time_col] = pd.to_datetime(df[time_col], utc=False)
df = df.sort_values(time_col).reset_index(drop=True)

for lag_h in [24, 168]:
    df[f"lag_{lag_h}"] = df[target_col].shift(lag_h)

work_df = df.dropna(subset=["lag_24", "lag_168", target_col]).reset_index(drop=True)

split_idx = int(len(work_df) * train_fraction)
train_df = work_df.iloc[:split_idx].copy()
test_df = work_df.iloc[split_idx:].copy()

print(f"Rows total: {len(work_df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


Rows total: 17,376
Train rows: 13,900
Test rows: 3,476


In [4]:
# Point forecast baselines (sklearn only)
model_specs = {
    "naive_24": ["lag_24"],
    "naive_168": ["lag_168"],
    "naive_24_168_combined": ["lag_24", "lag_168"],
}

fit_intercept = bool(cfg.get("point_models", {}).get("linear_regression", {}).get("fit_intercept", True))
results = []

for name, feature_cols in model_specs.items():
    model = LinearRegression(fit_intercept=fit_intercept)
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df[target_col]
    y_test = test_df[target_col]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append(
        {
            "model": name,
            "features": ", ".join(feature_cols),
            "mae": mean_absolute_error(y_test, preds),
            "rmse": root_mean_squared_error(y_test, preds),
            "r2": r2_score(y_test, preds),
        }
    )

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df


,model,features,mae,rmse,r2
0,naive_24_168_combined,"lag_24, lag_168",8654.757159,10858.532030,0.414516
1,naive_24,lag_24,8964.031023,11273.969464,0.368859
2,naive_168,lag_168,11035.724277,13604.348060,0.080973


### Notes
- This notebook is intentionally minimal and does not save outputs/artifacts.
- Quantile models and LightGBM workflows remain in `02_model_training.ipynb`.
